&nbsp;

# Prepare Gene Ontology - gene (transcript) Association File

- `GOAtools` で使用するための association file を作成する

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_contents
from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

## QuickGO annotation (human all)

- `NOT`を含むアノテーションは、今回考慮しない

In [2]:
quickgo_human_all = pl.read_csv(
    "../../Data/Database/QuickGO/QuickGO-annotations-human-all-1792593337498-20261021.tsv.gz",
    # QuickGO export is tab-separated. Default comma split treats the header as one field.
    separator="\t"
).filter(
    # "|" is literal. As a regex, "NOT|" matches every string.
    ~(pl.col("QUALIFIER").str.contains("NOT|", literal=True)) # 今回はこの NOT を含むアノテーションを考慮しない
)

print(quickgo_human_all.select("GENE PRODUCT DB").unique())

pl.Config.set_tbl_rows(22)
print(quickgo_human_all.select("QUALIFIER").sort("QUALIFIER").unique())

pl.Config.set_tbl_rows(2)
display(quickgo_human_all)

shape: (2, 1)
┌─────────────────┐
│ GENE PRODUCT DB │
│ ---             │
│ str             │
╞═════════════════╡
│ UniProtKB       │
│ RNAcentral      │
└─────────────────┘
shape: (13, 1)
┌─────────────────────────────────┐
│ QUALIFIER                       │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ acts_upstream_of                │
│ acts_upstream_of_negative_effe… │
│ acts_upstream_of_or_within      │
│ acts_upstream_of_or_within_neg… │
│ acts_upstream_of_or_within_pos… │
│ acts_upstream_of_positive_effe… │
│ colocalizes_with                │
│ contributes_to                  │
│ enables                         │
│ involved_in                     │
│ is_active_in                    │
│ located_in                      │
│ part_of                         │
└─────────────────────────────────┘


GENE PRODUCT DB,GENE PRODUCT ID,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""UniProtKB""","""A0A023HJ61""","""RAB4A""","""enables""","""GO:0003924""","""GTPase activity""","""ECO:0000501""","""IEA""","""GO_REF:0000120""","""InterPro:IPR001806|UniRule:UR0…",9606,"""UniProt""",null,20260727,"""HRES-1/RAB4 variant""","""RAB4A""","""protein""",null,"""F"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""RNAcentral""","""URS0000EF3174_9606""","""Homo sapiens (human) WEE2 anti…","""enables""","""GO:0030627""","""pre-mRNA 5'-splice site bindin…","""ECO:0000256""","""IEA""","""GO_REF:0000115""","""Rfam:RF00003""",9606,"""RNAcentral""",null,20260727,"""Homo sapiens (human) WEE2 anti…",null,"""RNA""",null,"""F"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## (1) BackGround Transcript List

- 今回は、GENCODE release 49 の全ての transcript ではなく、salmon で発現定量できた transcript list をクエリとする

&nbsp;

&nbsp;

## All quantifed Transcripts in this study

- より厳密に背景データを考えるために、今回は、発現定量をすることができた遺伝子を対象に association file を作成する

In [3]:
HNscore_all_transcript = pl.read_parquet(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/annotation/HNscore_gencode_human_261001_HN2_transcript_all_gencode_annotation.parquet"
)

display(HNscore_all_transcript)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000560179.2""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]","""-""",1,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""RNA-Seq supported partial""]",1


In [4]:
background_transcript_list = HNscore_all_transcript.select(
    "transcript_id"
).with_columns(
    # Drop the Ensembl version suffix, e.g. ENST00000791171.1 -> ENST00000791171
    pl.col("transcript_id").str.replace(r"\.\d+$", "")
)

background_transcript_list.write_csv(
    "../../out/enrichment_analysis/background/background_quantified_transcript_list.txt",
    separator="\t",
    include_header=False
)

display(background_transcript_list)

transcript_id
str
"""ENST00000309758"""
…
"""ENST00000560179"""


In [5]:
# protein-coding だけの background を作成する
background_pc_transcript_list = HNscore_all_transcript.filter(
    pl.col("transcript_type") == "protein_coding"
).select(
    pl.col("transcript_id")
).with_columns(
    pl.col("transcript_id").str.replace(r"\.\d+$", "")
).unique()

background_pc_transcript_list.write_csv(
    "../../out/enrichment_analysis/background/background_quantified_pc_transcript_list.txt",
    separator="\t",
    include_header=False
)

display(background_pc_transcript_list)

transcript_id
str
"""ENST00000616425"""
…
"""ENST00000495019"""


In [6]:
# lncRNA だけの background を作成する
background_nc_transcript_list = HNscore_all_transcript.filter(
    pl.col("transcript_type") == "lncRNA"
).select(
    pl.col("transcript_id")
).with_columns(
    pl.col("transcript_id").str.replace(r"\.\d+$", "")
).unique()

background_nc_transcript_list.write_csv(
    "../../out/enrichment_analysis/background/background_quantified_lncRNA_transcript_list.txt",
    separator="\t",
    include_header=False
)

display(background_nc_transcript_list)

transcript_id
str
"""ENST00000416521"""
…
"""ENST00000640157"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## (2) Transcript - GO association file

- この association file は、必ずしも、背景の transcript と数をあわせる必要はなく、あくまで包摂していることが重要

In [7]:
all_transcript = pl.read_csv(
    "../../Data/Database/GENCODE_v49/gencode_v49_tx2gene_chr_patch_hapl_scaff_v260909.tsv",
    separator="\t"
)

display(all_transcript)

transcript_id,gene_id
str,str
"""ENST00000832824.1""","""ENSG00000290825.2"""
…,…
"""ENST00000839853.1""","""ENSG00000309258.1"""


&nbsp;

&nbsp;

## TogoID process (2026-10-22)

In [ ]:
# # transcript list
# all_quantified_transcript_list = (
#     all_transcript
#     .select(pl.col("transcript_id"))
#     .unique()
# ).write_csv(
#     "../../out/enrichment_analysis/togoid/all_transcript_list.txt",
#     separator="\t",
#     include_header=False
# )

&nbsp;

&nbsp;

## Ensembl Transcript ID - Ensembl Protein ID - UniProt ID

- TogoID access: 2026-10-22
- `Ensembl transcript - is translated to - Ensembl protein - is equivalent to - UniProt`
- command: `bash ./scripts/togoid_convert.sh ./out/enrichment_analysis/togoid/all_transcript_list.txt ensembl_transcript,ensembl_protein,uniprot ./out/enrichment_analysis/togoid/all_transcript_uniprot.tsv`

In [9]:
uniprot_convert = pl.read_csv(
    "../../out/enrichment_analysis/togoid/all_transcript_uniprot.tsv.gz",
    separator="\t"
).rename(
    {
        "uniprot_id": "GENE PRODUCT ID"
    }
)

display(uniprot_convert)

ensembl_transcript_id,ensembl_protein_id,GENE PRODUCT ID
str,str,str
"""ENST00000886743""","""ENSP00000556802""","""A0ACI8SMA2"""
…,…,…
"""ENST00000445230""",null,null


In [10]:
uniprot_convert_filtered = uniprot_convert.filter(
    pl.col("GENE PRODUCT ID").is_not_null()
)

display(uniprot_convert_filtered)

ensembl_transcript_id,ensembl_protein_id,GENE PRODUCT ID
str,str,str
"""ENST00000886743""","""ENSP00000556802""","""A0ACI8SMA2"""
…,…,…
"""ENST00000908583""","""ENSP00000578642""","""Q13421"""


&nbsp;

&nbsp;

## Protein coding transcript - GO association

- まずは、UniProt ID をキーとして、対応付ける
- また、UniProtKB 由来の association file を作成し、これをバックグラウンドにする

In [11]:
protein_coding_transcript_go = uniprot_convert_filtered.join(
    quickgo_human_all,
    on="GENE PRODUCT ID",
    how="inner"
).drop(
    "ensembl_protein_id"
).rename(
    {
        "ensembl_transcript_id": "transcript_id"
    }
)

display(protein_coding_transcript_go)

transcript_id,GENE PRODUCT ID,GENE PRODUCT DB,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""ENST00000612959""","""A0A024QZ33""","""UniProtKB""","""NSRP1""","""involved_in""","""GO:0000381""","""regulation of alternative mRNA…","""ECO:0000256""","""IEA""","""GO_REF:0000002""","""InterPro:IPR018612""",9606,"""InterPro""",null,20260727,"""Nuclear speckle splicing regul…","""NSRP1""","""protein""",null,"""P"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000610140""","""X6RM59""","""UniProtKB""","""NT5C3A""","""located_in""","""GO:0005783""","""endoplasmic reticulum""","""ECO:0000314""","""IDA""","""GO_REF:0000052""",null,9606,"""HPA""",null,20251115,"""5'-nucleotidase""","""NT5C3A""","""protein""",null,"""C"""


&nbsp;

&nbsp;

&nbsp;

## Non-coding Transcript - GO association

- Transcript ID のバージョンを落とす
- RNACentral ID に 「_9606」をつける

In [18]:
transcript_rnacentral_id = pl.read_csv(
    "../../Data/Database/RNACentral/release27/database_mapping/ensembl_gencode.tsv",
    separator="\t",
    has_header=False,
    new_columns=[
        "RNACentral_ID",
        "relationship",
        "transcript_id",
        "taxon_id",
        "type",
        "gene_id"
    ]
).filter(
    pl.col("taxon_id") == 9606
).select(
    pl.col("transcript_id"),
    pl.col("RNACentral_ID").alias("GENE PRODUCT ID")
).with_columns(
    # Append the human NCBI taxon id used by RNAcentral, e.g. URS0002A62C26_9606
    pl.concat_str(pl.col("GENE PRODUCT ID"), pl.lit("_9606")).alias("GENE PRODUCT ID"),
)

display(transcript_rnacentral_id)

transcript_id,GENE PRODUCT ID
str,str
"""ENST00000585414""","""URS0000000055_9606"""
…,…
"""ENST00000787161""","""URS000344EFF2_9606"""


In [19]:
non_coding_transcript_go = transcript_rnacentral_id.join(
    quickgo_human_all,
    on="GENE PRODUCT ID",
    how="inner"
)

display(non_coding_transcript_go)

transcript_id,GENE PRODUCT ID,GENE PRODUCT DB,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""ENST00000469289""","""URS000000192A_9606""","""RNAcentral""","""Homo sapiens (human) MIR1302-2…","""involved_in""","""GO:0035195""","""miRNA-mediated post-transcript…","""ECO:0000256""","""IEA""","""GO_REF:0000115""","""Rfam:RF00951""",9606,"""RNAcentral""",null,20260727,"""Homo sapiens (human) MIR1302-2…",null,"""RNA""",null,"""P"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000666407""","""URS0000EF3174_9606""","""RNAcentral""","""Homo sapiens (human) WEE2 anti…","""enables""","""GO:0030627""","""pre-mRNA 5'-splice site bindin…","""ECO:0000256""","""IEA""","""GO_REF:0000115""","""Rfam:RF00003""",9606,"""RNAcentral""",null,20260727,"""Homo sapiens (human) WEE2 anti…",null,"""RNA""",null,"""F"""


&nbsp;

&nbsp;

&nbsp;

## (IMPORTANT!) 整合性 check

- [URS0002A5FA5F_9606](https://rnacentral.org/rna/URS0002A5FA5F/9606) みたいに、QuickGOを見ると Rfam にアノテーションされているが、RNACentral を見ると、アノテーションがない、という齟齬が見られることから、両方のソースをみて check する
- RNACentral の release より QuickGO が遅れてアノテーションされている？

In [20]:
rnacentral_annotation = pl.read_csv(
    "../../Data/Database/RNACentral/release27/rnacentral_rfam_annotations.tsv.gz",
    separator="\t",
    has_header=False,
    new_columns=[
       "RNACentral_ID",
        "GO TERM",
        "WITH/FROM"
    ],
    n_threads=16
).filter(
    # only human
    pl.col("RNACentral_ID").str.contains("_9606")
).rename(
    {
        "RNACentral_ID": "GENE PRODUCT ID"
    }
)

display(rnacentral_annotation)

GENE PRODUCT ID,GO TERM,WITH/FROM
str,str,str
"""URS0000001346_9606""","""GO:0030533""","""Rfam:RF00005"""
…,…,…
"""URS0003707711_9606""","""GO:0006396""","""Rfam:RF00277"""


In [21]:
# check
URS0000A77400_9606 = rnacentral_annotation.filter(
    pl.col("GENE PRODUCT ID") == "URS0000A77400_9606"
)

display(URS0000A77400_9606)

GENE PRODUCT ID,GO TERM,WITH/FROM
str,str,str
"""URS0000A77400_9606""","""GO:2000147""","""Rfam:RF01871"""


&nbsp;

&nbsp;

### RNACentral 以外の アノテーションは残しておく

In [22]:
# Keep non-RNAcentral assignments as they are.
# Only RNAcentral rows are checked against the release-27 Rfam table.
non_coding_transcript_go_rnacentral = non_coding_transcript_go.filter(
    pl.col("ASSIGNED BY") == "RNAcentral"
)
non_coding_transcript_go_other = non_coding_transcript_go.filter(
    pl.col("ASSIGNED BY") != "RNAcentral"
)

non_coding_transcript_go_rnacentral_checked = non_coding_transcript_go_rnacentral.join(
    rnacentral_annotation,
    on=[
        "GENE PRODUCT ID",
        "GO TERM",
        "WITH/FROM"
    ],
    how="inner"
)

non_coding_transcript_go_rnacentral_validated = pl.concat(
    [
        non_coding_transcript_go_rnacentral_checked,
        non_coding_transcript_go_other
    ],
    how="vertical_relaxed"
).sort(
    "transcript_id",
    "GO TERM"
)

display(non_coding_transcript_go_rnacentral_validated)

transcript_id,GENE PRODUCT ID,GENE PRODUCT DB,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""ENST00000362512""","""URS00006ECA78_9606""","""RNAcentral""","""Homo sapiens (human) RNA, U12 …","""involved_in""","""GO:0000348""","""mRNA branch site recognition""","""ECO:0000256""","""IEA""","""GO_REF:0000115""","""Rfam:RF00007""",9606,"""RNAcentral""",null,20260727,"""Homo sapiens (human) RNA, U12 …",null,"""RNA""",null,"""P"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000717968""","""URS0000D5CBB3_9606""","""RNAcentral""","""Homo sapiens (human) nuclear p…","""located_in""","""GO:0042382""","""paraspeckles""","""ECO:0000256""","""IEA""","""GO_REF:0000115""","""Rfam:RF01955|Rfam:RF01956|Rfam…",9606,"""RNAcentral""",null,20260727,"""Homo sapiens (human) nuclear p…",null,"""RNA""",null,"""C"""


&nbsp;

&nbsp;

&nbsp;

## Concat Process

In [23]:
all_go_association = pl.concat(
    [
        protein_coding_transcript_go,
        non_coding_transcript_go_rnacentral_validated
    ],
    how="vertical_relaxed"
)

display(all_go_association)

transcript_id,GENE PRODUCT ID,GENE PRODUCT DB,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""ENST00000612959""","""A0A024QZ33""","""UniProtKB""","""NSRP1""","""involved_in""","""GO:0000381""","""regulation of alternative mRNA…","""ECO:0000256""","""IEA""","""GO_REF:0000002""","""InterPro:IPR018612""",9606,"""InterPro""",null,20260727,"""Nuclear speckle splicing regul…","""NSRP1""","""protein""",null,"""P"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000717968""","""URS0000D5CBB3_9606""","""RNAcentral""","""Homo sapiens (human) nuclear p…","""located_in""","""GO:0042382""","""paraspeckles""","""ECO:0000256""","""IEA""","""GO_REF:0000115""","""Rfam:RF01955|Rfam:RF01956|Rfam…",9606,"""RNAcentral""",null,20260727,"""Homo sapiens (human) nuclear p…",null,"""RNA""",null,"""C"""


In [25]:
all_go_association.write_csv(
    "../../out/enrichment_analysis/transcript_go_association/all_transcript_go_relation_quickgo.tsv.gz",
    separator="\t",
    compression="gzip",
    compression_level=9
)

&nbsp;

&nbsp;

## (2) Create Transcript - GO association File

In [26]:
transcript_go_association = all_go_association.select(
    "transcript_id",
    "GO TERM"
).group_by(
    "transcript_id"
).agg(
    # unique() is list[str] only after agg finishes.
    pl.col("GO TERM").unique().alias("GO term accessions")
).with_columns(
    # Join whole GO ids. map_elements would join the characters of each id.
    pl.col("GO term accessions").list.sort().list.join(";")
).sort(
    "transcript_id"
)

display(transcript_go_association)

transcript_id,GO term accessions
str,str
"""ENST00000000233""","""GO:0003924;GO:0005515;GO:00055…"
…,…
"""ENST00000973046""","""GO:0003676;GO:0003723;GO:00055…"


In [27]:
transcript_go_association.write_csv(
    "../../out/enrichment_analysis/transcript_go_association/transcript_go_association.txt",
    separator="\t",
    include_header=False
)

&nbsp;

&nbsp;

&nbsp;

## (3) up5p query transcript list

- protein coding transcript だけのリストも作成する
- version は取り除く

In [28]:
top5_transcript_list = pl.read_csv(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/annotation/HN2_top5p_transcripts_annotation_261001.tsv",
    separator="\t"
).with_columns(
    # Drop the Ensembl version suffix, e.g. ENST00000791171.1 -> ENST00000791171
    pl.col("transcript_id").str.replace(r"\.\d+$", "")
)

top5_transcript_list.select(
    "transcript_id"
).unique().write_csv(
    "../../out/enrichment_analysis/query/up5p_transcript_list.txt",
    separator="\t",
    include_header=False
)

display(top5_transcript_list)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,representative_annotation_remarks,representative_transcript_type
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,i64,str,str,str,str,str,i64,str,str
"""ENST00000309758""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","""basic, Ensembl_canonical, GENC…","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","""MANE_select""",1,"""MANE_select""","""protein_coding"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000774581""",101,57,248,44,34,44,173,-10,null,null,"""lncRNA""","""ENSG00000300853.1""","""basic, TAGENE""","""+""",2,"""URS0002A25289""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","""CLS3 project""",1,"""derived from CLS3 project""","""lncRNA"""


In [29]:
# protein-coding だけのリストも作成する

up5p_pc_transcript_list = top5_transcript_list.filter(
    (pl.col("transcript_type") == "protein_coding")
).select(
    "transcript_id"
).unique()

up5p_pc_transcript_list.write_csv(
    "../../out/enrichment_analysis/query/up5p_pc_transcript_list.txt",
    separator="\t",
    include_header=False
)

display(up5p_pc_transcript_list)

transcript_id
str
"""ENST00000314222"""
…
"""ENST00000524699"""


In [30]:
# lncRNA だけのリストも作成する
up5p_lncRNA_transcript_list = top5_transcript_list.filter(
    pl.col("transcript_type") == "lncRNA"
).select(
    "transcript_id"
).unique()

up5p_lncRNA_transcript_list.write_csv(
    "../../out/enrichment_analysis/query/up5p_lncRNA_transcript_list.txt",
    separator="\t",
    include_header=False
)

display(up5p_lncRNA_transcript_list)

transcript_id
str
"""ENST00000660265"""
…
"""ENST00000636036"""


&nbsp;

&nbsp;

## (4) down5p transcript list

- protein-coding transcript だけのリストもここで作成しておく
- バージョン情報は取り除く

In [31]:
down5p_transcript_list = pl.read_csv(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/annotation/HN2_bottom5p_transcripts_annotation_261001.tsv",
    separator="\t"
).with_columns(
    pl.col("transcript_id").str.replace(r"\.\d+$", "")
)

down5p_transcript_list.select(
    "transcript_id"
).unique().write_csv(
    "../../out/enrichment_analysis/query/down5p_transcript_list.txt",
    separator="\t",
    include_header=False
)

display(down5p_transcript_list)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,representative_annotation_remarks,representative_transcript_type
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,i64,str,str,str,str,str,i64,str,str
"""ENST00000535641""",84,132,190,-48,58,64,129,-6,"""ANAPC5""","""HGNC:15713""","""retained_intron""","""ENSG00000089053.14""","""""","""-""",0,null,null,"""polya_signal, polya_site""","""ensembl_havana_gene_homo_sapie…",null,0,"""no annotation remark""","""others"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000560179""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""","""retained_intron""","""ENSG00000140374.18""","""RNA_Seq_supported_partial""","""-""",1,null,null,null,"""ensembl_havana_gene_homo_sapie…","""RNA-Seq supported partial""",1,"""Other""","""others"""


In [33]:
# protein-coding だけのリストも作成する
down5p_pc_transcript_list = down5p_transcript_list.filter(
    (pl.col("transcript_type") == "protein_coding")
).select(
    "transcript_id"
).unique().write_csv(
    "../../out/enrichment_analysis/query/down5p_pc_transcript_list.txt",
    separator="\t",
    include_header=False
)

In [34]:
# lncRNA だけのリストも作成する
down5p_lncRNA_transcript_list = down5p_transcript_list.filter(
    pl.col("transcript_type") == "lncRNA"
).select(
    "transcript_id"
).unique()

down5p_lncRNA_transcript_list.write_csv(
    "../../out/enrichment_analysis/query/down5p_lncRNA_transcript_list.txt",
    separator="\t",
    include_header=False
)

display(down5p_lncRNA_transcript_list)

transcript_id
str
"""ENST00000721413"""
…
"""ENST00000442532"""


&nbsp;

&nbsp;

&nbsp;

## (5) `go.obo` file Download (remark)

- command: `curl -L -o ./Data/Database/GO/go_release_20260805.obo http://release.geneontology.org/2026-08-05/ontology/go.obo`